In [23]:
import os
os.chdir('/users/sgdbareh/volatile/ECHR_Importance/BERT-rerank')

In [24]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datetime import datetime
import logging
import math
from torch.utils.data import DataLoader, Subset
from sentence_transformers import LoggingHandler, util
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from sentence_transformers.readers import InputExample
import torch
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from utils import EarlyStopping, calculate_mcc, custom_collate_fn, CrossEncoderMod
from torch.amp import GradScaler, autocast
from optparse import OptionParser


### hmm

In [34]:
data = pd.read_pickle('/users/sgdbareh/volatile/ECHR_Importance/Art_3_Data_Process/comm_cases_with_metadata_UPDATE.pkl')
data_2 = pd.read_pickle('/users/sgdbareh/volatile/ECHR_Importance/Art_3_Data_Process/outcome_cases.pkl')

In [35]:
data_2

,Facts,Word Count,The Law,date,File,appno,doctypebranch,respondent,extractedappno,conclusion,importance,kpthesaurus,sclappnos
0,"The applicant, born in 1956, is a Turkish citi...",272,1. The applicant complains under Article 5 par...,1997-12-09,001-67704,31859/96,ADMISSIBILITY,TUR,31859/96;22761/93;24722/94;1994/95,Inadmissible,4,448;45;350,22761/93;24722/94
1,The applicant is a British citizen born in 195...,850,1. The applicant complains that the reviews by...,1998-12-01,001-4840,40787/98,ADMISSIBILITY,GBR,40787/98;20448/92;15882/89,Partly inadmissible,4,451;429;203;268;329;376;350,15882/89
2,The applicant is a British citizen born in 196...,2233,The applicant complains that he has not been a...,1998-12-01,001-4894,32340/96,ADMISSIBILITY,GBR,32340/96,Admissible,4,448;438;45;350;192;90,
3,"The applicant, born in 1955, is a citizen of S...",803,The applicant complains under Article 3 of the...,1998-12-08,001-4497,44667/98,ADMISSIBILITY,DEU,44667/98,Inadmissible,4,14;128;350;193,
4,"The applicant, born in 1974, is a citizen of M...",935,1. The applicant complains that his deportatio...,1999-01-19,001-4507,42367/98,ADMISSIBILITY,SWE,42367/98,Inadmissible,3,350;492;193;90;192;89,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7666,10. The applicant was born in 1985 and now res...,12338,ALLEGED VIOLATION OF ARTICLE 3 OF THE CONVENTI...,2021-12-07,001-214330,57467/15,GRANDCHAMBER,DNK,57467/15;578/16;353/16;41738/10;17299/12;26565...,No violation of Article 3 - Prohibition of tor...,1,350;620;451;628;429;268;329;333;216;577;283;23...,47486/06;17299/12;3138/16;65550/13;44599/98;23...
7667,THE CIRCUMSTANCES OF THE CASE11. The applicant...,10107,ALLEGED VIOLATION OF ARTICLE 3 OF THE CONVENTI...,2022-04-29,001-217061,28492/15;49975/15,GRANDCHAMBER,RUS,28492/15;49975/15;77658/11;49747/11;71386/10;2...,No violation of Article 3 - Prohibition of tor...,1,350;618,60342/16;12148/18;46240/15;68377/17;530/18;123...
7668,"THE CIRCUMSTANCES OF THE CASE6. The applicant,...",6764,ALLEGED VIOLATION OF ARTICLE 3 OF THE CONVENTI...,2022-11-03,001-220484,22854/20,GRANDCHAMBER,GBR,22854/20;140/10;66069/09;9146/07;32650/07;2190...,Struck out of the list (Art. 37) Striking out ...,1,350;89;618;192;609;478,35763/97;43611/11;20672/15;9146/07;32650/07;36...
7669,1. The case originated in an application (no. ...,1131,Legal Framework12. Article 38 of the Conventio...,2023-04-28,001-224629,38263/08,GRANDCHAMBER,RUS,38263/08;59532/00;10865/09;40792/10;8019/16;13...,Non-pecuniary damage - award (Article 41 - Non...,1,216;283;577;121;594;26;126;449;595;231;496;350...,46454/11;59532/00;20914/07;13216/05;25781/94;4...


In [24]:
data['extractedapplen'].head(50)

0      2
1     11
2      0
3     33
4     12
5      3
6     17
7     19
8      0
9     17
10    12
11     0
12    10
13    21
14    31
15    43
16    15
17     0
18    12
19    12
20    16
21    21
22    11
23     1
24    38
25     0
26    25
27    12
28     0
29     8
30    18
31     0
32    18
33     1
34    13
35    60
36    23
37    35
38    28
39    34
40    19
41     5
42    19
43     9
44     0
45     0
46    21
47     0
48    33
49     7
Name: extractedapplen, dtype: int64

In [23]:
data_2['extractedapplen'].head(50)

0      2
1     11
2      0
3     33
4     12
5      3
6     13
7     19
8      0
9     17
10    12
11     0
12    10
13    21
14    31
15    41
16    15
17     0
18    12
19    12
20    15
21    21
22    10
23     1
24    38
25     0
26    25
27    12
28     0
29     8
30    18
31     0
32    18
33     1
34    12
35    55
36    23
37    35
38    28
39    34
40    19
41     5
42    19
43     9
44     0
45     0
46    21
47     0
48    33
49     7
Name: extractedapplen, dtype: int64

### BERT Test

In [3]:
#### Just some code to print debug information to stdout
logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO, handlers=[LoggingHandler()]
)
logger = logging.getLogger(__name__)
#### /print debug information to stdout


In [5]:
model_save_path = "/users/sgdbareh/volatile/ECHR_Importance/BERT-rerank/model" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
use_cuda = torch.cuda.is_available()

train = pd.read_pickle('/users/sgdbareh/volatile/ECHR_Importance/BERT-rerank/BERT_training_data_df.pkl')

label2int = {"neg": 0, "pos": 1}

train_samples = []

# Iterate over the DataFrame rows
for index, row in train.iterrows():
    # Create InputExample for positive label
    train_samples.append(InputExample(texts=[str(row['Comm_Case']).lower(), str(row['Positive']).lower()], label=label2int["pos"]))
    # Create InputExample for negative label
    train_samples.append(InputExample(texts=[str(row['Comm_Case']).lower(), str(row['Negative']).lower()], label=label2int["neg"]))

# Split the data into training and test sets
train_samples, test_samples = train_test_split(train_samples, test_size=0.2, random_state=456)



In [8]:
# Evaluate the final model on the test set
test_dataloader = DataLoader(test_samples, shuffle=False, batch_size=16)
evaluator = CEBinaryClassificationEvaluator.from_input_examples(test_samples, name='Relevance BERT')

# Load the final model for evaluation
final_model = CrossEncoder(f"/users/sgdbareh/volatile/ECHR_Importance/BERT-rerank/model2024-11-07_14-39-42_FINAL")

test_evaluation_result = evaluator(final_model)
logger.info(f"Final Evaluation result: {test_evaluation_result}")




2024-11-07 15:14:35 - Use pytorch device: cuda
2024-11-07 15:14:35 - CEBinaryClassificationEvaluator: Evaluating the model on Relevance BERT dataset:
2024-11-07 15:15:16 - Accuracy:           80.97	(Threshold: 0.8159)
2024-11-07 15:15:16 - F1:                 79.87	(Threshold: 0.0312)
2024-11-07 15:15:16 - Precision:          76.84
2024-11-07 15:15:16 - Recall:             83.15
2024-11-07 15:15:16 - Average Precision:  88.01

2024-11-07 15:15:16 - Final Evaluation result: 0.8800779037629908


Batches:   0%|          | 0/30 [00:00<?, ?it/s]


TypeError: object of type 'InputExample' has no len()

In [18]:
evaluator = CEBinaryClassificationEvaluator.from_input_examples(test_samples, name='Relevance BERT')
test_MCC_data = evaluator.sentence_pairs

scores = final_model.predict(test_MCC_data)
scores

Batches: 100%|██████████| 30/30 [00:36<00:00,  1.22s/it]


array([0.98628145, 0.23627326, 0.9983015 , 0.99835265, 0.9981299 ,
       0.00186099, 0.00282638, 0.00963226, 0.99632746, 0.21033435,
       0.9859927 , 0.99894613, 0.99208844, 0.998917  , 0.98066187,
       0.00310433, 0.00408415, 0.00209924, 0.01124159, 0.00187133,
       0.00165371, 0.9963142 , 0.00357381, 0.99836034, 0.9988048 ,
       0.9387336 , 0.810339  , 0.00278707, 0.00338454, 0.5703521 ,
       0.00177042, 0.99873406, 0.29781067, 0.9976921 , 0.00447907,
       0.998898  , 0.00579973, 0.0018477 , 0.00635233, 0.00400014,
       0.01253772, 0.00601326, 0.00259347, 0.99902403, 0.99682885,
       0.99817944, 0.99733245, 0.9794597 , 0.99413353, 0.00196269,
       0.99685943, 0.9957944 , 0.00697753, 0.0017769 , 0.9502528 ,
       0.00224384, 0.9986829 , 0.05524009, 0.00186009, 0.99879885,
       0.9988881 , 0.00207974, 0.99884593, 0.9990476 , 0.00362875,
       0.81624234, 0.0028343 , 0.01511883, 0.99532866, 0.00351187,
       0.9975848 , 0.00206425, 0.99621177, 0.9989254 , 0.00925

In [19]:
# Calculate MCC
y_true = [example.label for example in test_samples]
y_pred = [int(score > 0.5) for score in final_model.predict(evaluator.sentence_pairs)]


Batches: 100%|██████████| 30/30 [00:36<00:00,  1.22s/it]


NameError: name 'calculate_mcc' is not defined

In [25]:
test_mcc = calculate_mcc(y_true, y_pred)
logger.info(f"Final model test MCC: {test_mcc}")

2024-11-07 15:25:39 - Final model test MCC: 0.6018637707740765


In [27]:
from sklearn.metrics import f1_score

f1_score(y_true, y_pred)

0.7900113507377979